In [ ]:
import pandas as pd
data = pd.read_csv('https://raw.githubusercontent.com/dsrscientist/dataset1/master/heart_disease.csv')
data.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [ ]:
# ============================================================
#  BINARY CLASSIFICATION — Heart Disease (Tabular)
#  Dataset : UCI Heart Disease
#  Model   : MLP
#  UI      : Gradio
#
#  HOW TO RUN
#  ----------
#  Step 1 → Run BLOCK 1 to train and save model.pth
#  Step 2 → Run BLOCK 2 to launch the Gradio UI
#
#  pip install torch gradio scikit-learn pandas seaborn matplotlib
# ============================================================


# ╔══════════════════════════════════════════════════════════════╗
# ║  —> TRAINING & SAVING MODEL WEIGHTS                 ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Imports ──────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)

# ── Config ───────────────────────────────────────────────────
EPOCHS        = 2           # ← change epochs
BATCH_SIZE    = 32          # ← change batch size
LEARNING_RATE = 1e-3        # ← change learning rate
TEST_SIZE     = 0.2
MODEL_PATH    = "heart_model.pth"       # weights saved here
SCALER_PATH   = "heart_scaler.pkl"      # scaler saved here (needed for UI)
FEATURE_PATH  = "heart_features.pkl"    # feature names saved here
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Load Data ────────────────────────────────────────────────
url = ("https://raw.githubusercontent.com/dsrscientist/"
       "dataset1/master/heart_disease.csv")
try:
    df = pd.read_csv(url)
    df.columns = ["age","sex","cp","trestbps","chol","fbs","restecg",
                  "thalach","exang","oldpeak","slope","ca","thal","target"]
except Exception:
    from sklearn.datasets import fetch_openml
    data = fetch_openml("heart", version=1, as_frame=True, parser="auto")
    df   = data.frame.copy()
    df.columns = [c.lower().replace(" ","_") for c in df.columns]
    df["target"] = (df["target"].astype(str).str.strip() != "1").astype(int)

# ── Feature Engineering (Tabular) ────────────────────────────
df = df.replace("?", np.nan).apply(pd.to_numeric, errors="coerce")
for col in df.columns:
    df[col].fillna(df[col].median(), inplace=True)

df["target"]           = (df["target"] > 0).astype(int)
df                     = pd.get_dummies(df, columns=["cp","restecg","slope","thal"], drop_first=True)
df["age_group"]        = pd.cut(df["age"], bins=[0,40,55,70,100], labels=[0,1,2,3]).astype(int)
df["hr_ratio"]         = df["thalach"] / (df["trestbps"] + 1e-5)
df["high_chol"]        = (df["chol"] > 240).astype(int)

FEATURE_COLS = [c for c in df.columns if c != "target"]
X = df[FEATURE_COLS].values.astype(np.float32)
y = df["target"].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42, stratify=y)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Save scaler and feature names for the UI block
joblib.dump(scaler, SCALER_PATH)
joblib.dump(FEATURE_COLS, FEATURE_PATH)
print(f"Scaler saved → {SCALER_PATH}")
print(f"Feature list saved → {FEATURE_PATH}")

X_tr = torch.tensor(X_train, dtype=torch.float32)
X_te = torch.tensor(X_test,  dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.float32)
y_te = torch.tensor(y_test,  dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=BATCH_SIZE, shuffle=False)

# ── Model ────────────────────────────────────────────────────
class BinaryMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            # ── LAYER 1 ──────────────────────────────────────
            nn.Linear(input_dim, 64),   # ← change units: 128, 256
            nn.BatchNorm1d(64),         # ← remove to disable BatchNorm
            nn.ReLU(),                  # ← swap: nn.Tanh() | nn.LeakyReLU(0.1) | nn.GELU()
            nn.Dropout(0.3),            # ← change dropout rate
            # ── LAYER 2 ──────────────────────────────────────
            nn.Linear(64, 32),          # ← change units
            nn.BatchNorm1d(32),         # ← remove to disable BatchNorm
            nn.ReLU(),                  # ← swap activation
            nn.Dropout(0.3),            # ← change dropout rate
            # ── ADD LAYERS HERE ──────────────────────────────
            # nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.2),
            # ── OUTPUT ───────────────────────────────────────
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

model = BinaryMLP(input_dim=X_train.shape[1]).to(DEVICE)

# ── Loss & Optimizer ─────────────────────────────────────────
criterion = nn.BCELoss()                                        # ← Binary Cross-Entropy
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)   # ← swap: optim.SGD | optim.RMSprop

# ── Training Loop ────────────────────────────────────────────
train_losses, train_accs = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        out  = model(X_b)
        loss = criterion(out, y_b)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_b.size(0)
        correct += ((out >= 0.5).long() == y_b.long()).sum().item()
        total   += y_b.size(0)
    train_losses.append(running_loss / total)
    train_accs.append(correct / total)
    print(f"Epoch [{epoch}/{EPOCHS}]  Loss: {train_losses[-1]:.4f}  Acc: {train_accs[-1]:.4f}")

# ── Save Model Weights ───────────────────────────────────────
torch.save({
    "model_state_dict": model.state_dict(),
    "input_dim"       : X_train.shape[1],
    "feature_cols"    : FEATURE_COLS,
}, MODEL_PATH)
print(f"\nModel weights saved → {MODEL_PATH}")

# ── Evaluation ───────────────────────────────────────────────
model.eval()
all_preds, all_probs, all_labels = [], [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        probs = model(X_b.to(DEVICE)).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend((probs >= 0.5).astype(int))
        all_labels.extend(y_b.numpy())

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs)

print("\n── Classification Report ──")
print(classification_report(all_labels, all_preds,
                             target_names=["No Disease","Disease"]))
print(f"F1 : {f1_score(all_labels, all_preds):.4f}")
print(f"AUC: {roc_auc_score(all_labels, all_probs):.4f}")

# ── Training Plots ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Binary Classification — Heart Disease", fontsize=13, fontweight="bold")
c1, c2 = "#2471A3", "#C0392B"

ax = axes[0]
ax.plot(range(1, EPOCHS+1), train_losses, marker="o", color=c1)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss", color=c1)
ax.tick_params(axis="y", labelcolor=c1); ax.set_xticks(range(1, EPOCHS+1))
ax2 = ax.twinx()
ax2.plot(range(1, EPOCHS+1), train_accs, marker="s", color=c2)
ax2.set_ylabel("Accuracy", color=c2); ax2.tick_params(axis="y", labelcolor=c2)
ax.set_title("Loss & Accuracy")

cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1],
            xticklabels=["No Disease","Disease"],
            yticklabels=["No Disease","Disease"])
axes[1].set_title("Confusion Matrix")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")

fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc_val = roc_auc_score(all_labels, all_probs)
axes[2].plot(fpr, tpr, color=c1, lw=2, label=f"AUC = {auc_val:.3f}")
axes[2].plot([0,1],[0,1],"k--",lw=1)
axes[2].set_xlabel("FPR"); axes[2].set_ylabel("TPR")
axes[2].set_title("ROC Curve"); axes[2].legend()

plt.tight_layout()
plt.savefig("heart_training_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plots saved → heart_training_plots.png")

# Feature correlation heatmap
fig2, ax = plt.subplots(figsize=(10, 7))
corr_cols = ["age","trestbps","chol","thalach","oldpeak","hr_ratio","target"]
corr_cols = [c for c in corr_cols if c in df.columns]
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, ax=ax, linewidths=0.5)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("heart_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Heatmap saved → heart_correlation_heatmap.png")



In [ ]:

# ╔══════════════════════════════════════════════════════════════╗
# ║  —> LOAD WEIGHTS & LAUNCH GRADIO UI                 ║
# ║  Runs independently — no re-training needed                 ║
# ║  HuggingFace Spaces compatible (set share=False on Spaces)  ║
# ╚══════════════════════════════════════════════════════════════╝

import gradio as gr
import joblib, torch, torch.nn as nn, numpy as np, pandas as pd

# ── Paths (must match Block 1) ───────────────────────────────
MODEL_PATH   = "heart_model.pth"
SCALER_PATH  = "heart_scaler.pkl"
FEATURE_PATH = "heart_features.pkl"

# ── Recreate model architecture ──────────────────────────────
class BinaryMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),        nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 1),         nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

# ── Load saved weights & artefacts ───────────────────────────
checkpoint    = torch.load(MODEL_PATH, map_location="cpu")
input_dim     = checkpoint["input_dim"]
feature_cols  = checkpoint["feature_cols"]

model = BinaryMLP(input_dim)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded model from {MODEL_PATH}  (input_dim={input_dim})")

scaler = joblib.load(SCALER_PATH)
print(f"Loaded scaler from {SCALER_PATH}")

# ── Inference function ────────────────────────────────────────
def predict_heart_disease(age, sex, trestbps, chol, fbs,
                           thalach, exang, oldpeak, ca):
    """
    Takes raw user inputs, applies the same feature engineering
    used in training, scales, and returns prediction + probability.
    """
    # Build a minimal dataframe matching training features
    row = {
        "age":      age,
        "sex":      sex,
        "trestbps": trestbps,
        "chol":     chol,
        "fbs":      fbs,
        "thalach":  thalach,
        "exang":    exang,
        "oldpeak":  oldpeak,
        "ca":       ca,
        # categorical columns — use most common values as defaults
        "cp":       1,
        "restecg":  0,
        "slope":    1,
        "thal":     2,
    }
    df_row = pd.DataFrame([row])
    df_row = pd.get_dummies(df_row, columns=["cp","restecg","slope","thal"], drop_first=True)

    # Add derived features
    df_row["age_group"]  = int(pd.cut([age], bins=[0,40,55,70,100], labels=[0,1,2,3])[0])
    df_row["hr_ratio"]   = thalach / (trestbps + 1e-5)
    df_row["high_chol"]  = int(chol > 240)

    # Align columns to training feature set
    for col in feature_cols:
        if col not in df_row.columns:
            df_row[col] = 0
    df_row = df_row[feature_cols]

    X = scaler.transform(df_row.values.astype(np.float32))
    X_t = torch.tensor(X, dtype=torch.float32)

    with torch.no_grad():
        prob = model(X_t).item()

    label = "🔴 Heart Disease Likely" if prob >= 0.5 else "🟢 No Heart Disease"
    return f"{label}\n\nProbability of disease: {prob*100:.1f}%"

# ── Gradio Interface ──────────────────────────────────────────
with gr.Blocks(title="Heart Disease Predictor") as demo:
    gr.Markdown(
        "## 🫀 Heart Disease Risk Predictor\n"
        "MLP trained on UCI Heart Disease dataset. "
        "Enter patient details and click **Predict**."
    )
    with gr.Row():
        with gr.Column():
            age      = gr.Slider(20, 80,  value=50, step=1,   label="Age")
            sex      = gr.Radio([0, 1],   value=1,            label="Sex (0=Female, 1=Male)")
            trestbps = gr.Slider(80, 200, value=120, step=1,  label="Resting Blood Pressure (mmHg)")
            chol     = gr.Slider(100, 600,value=240, step=1,  label="Cholesterol (mg/dl)")
            fbs      = gr.Radio([0, 1],   value=0,            label="Fasting Blood Sugar > 120 (1=Yes)")
        with gr.Column():
            thalach  = gr.Slider(60, 220, value=150, step=1,  label="Max Heart Rate Achieved")
            exang    = gr.Radio([0, 1],   value=0,            label="Exercise Induced Angina (1=Yes)")
            oldpeak  = gr.Slider(0.0, 6.0,value=1.0,step=0.1, label="ST Depression (oldpeak)")
            ca       = gr.Slider(0, 4,   value=0,  step=1,   label="Number of Major Vessels (ca)")
            btn      = gr.Button("Predict", variant="primary")

    output = gr.Textbox(label="Prediction", lines=3)
    btn.click(fn=predict_heart_disease,
              inputs=[age, sex, trestbps, chol, fbs, thalach, exang, oldpeak, ca],
              outputs=output)

    gr.Markdown(
        "> **Note:** This is a demonstration model. "
        "Not for clinical use."
    )

# share=True  → public link  (use on local machine)
# share=False → local only   (use on HuggingFace Spaces — Spaces handles public URL)
demo.launch(share=False)